# LoRA-дообучение маленькой LLM для классификации заголовков — с честными бейзлайнами

**Задача:** отнести заголовок русской новости к одной из 6 рубрик — спорт, происшествия, политика, наука, культура, экономика.
**Данные:** `ai-forever/headline-classification` (ruMTEB, лицензия MIT).
**Модель:** `Qwen/Qwen2.5-1.5B-Instruct`, Colab T4.

**Порядок важен и не меняется:**
1. Смотрим на данные и проверяем утечку между train и test.
2. Фиксируем выборки — все варианты оцениваются на одних и тех же примерах.
3. Считаем бейзлайны **до** обучения: TF-IDF + логистическая регрессия, модель без примеров, модель с примерами.
4. Обучаем LoRA.
5. Сравниваем всё на одной тестовой выборке с доверительными интервалами и парным бутстрепом.



In [1]:
!pip -q uninstall -y torchao
!pip -q install peft

import os, re, json, random, time
from collections import Counter
import numpy as np
import torch
import transformers, peft, sklearn
from datasets import load_dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs("results", exist_ok=True)

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| sklearn", sklearn.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "НЕТ — включи T4")

torch 2.11.0+cu130 | transformers 5.17.0 | peft 0.21.0 | sklearn 1.6.1
GPU: Tesla T4


## 1. Данные

In [2]:
ds = load_dataset("ai-forever/headline-classification")
print(ds)

LABELS = ["спорт", "происшествия", "политика", "наука", "культура", "экономика"]
mapping = {}
for r in ds["train"].select(range(3000)):
    mapping.setdefault(r["label"], r["label_text"])
print("соответствие меток:", dict(sorted(mapping.items())))
assert [mapping[i] for i in range(6)] == LABELS, "порядок меток отличается — пришли мне этот вывод"

for split in ds:
    print(split, sorted(Counter(ds[split]["label_text"]).items()))
print("\nпримеры:")
for r in ds["train"].select(range(5)):
    print(f"  [{r['label_text']}] {r['text']}")

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 36000
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 12000
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 12000
    })
})
соответствие меток: {0: 'спорт', 1: 'происшествия', 2: 'политика', 3: 'наука', 4: 'культура', 5: 'экономика'}
train [('культура', 6000), ('наука', 6000), ('политика', 6000), ('происшествия', 6000), ('спорт', 6000), ('экономика', 6000)]
validation [('культура', 2000), ('наука', 2000), ('политика', 2000), ('происшествия', 2000), ('спорт', 2000), ('экономика', 2000)]
test [('культура', 2000), ('наука', 2000), ('политика', 2000), ('происшествия', 2000), ('спорт', 2000), ('экономика', 2000)]

примеры:
  [спорт] Зубков и Воевода захватили лидерство на ЧМ
  [политика] Россия заблокировала расширение миссии ОБСЕ на украинско-российской границе
  [проис

## 2. Проверка утечки между train и test

Источник датасета — корпус **парафраз**. Значит, один новостной повод может лежать несколькими похожими заголовками в разных сплитах. Если такие пары разъезжаются между обучением и проверкой, метрика растёт без улучшения модели — та же история, что с перепечатками в Спортмастере.

Проверяем два уровня:
- **точные дубли** после нормализации;
- **почти-дубли** — ближайший сосед из train для каждого test-заголовка по косинусу TF-IDF на символьных 3–5-граммах.

In [3]:
def norm(t):
    t = t.lower().replace("ё", "е")
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

tr_text, te_text = ds["train"]["text"], ds["test"]["text"]
tr_y, te_y = np.array(ds["train"]["label"]), np.array(ds["test"]["label"])
tr_n, te_n = [norm(t) for t in tr_text], [norm(t) for t in te_text]

s_tr = set(tr_n)
print("test, точно совпадающих с train:", sum(t in s_tr for t in te_n), "из", len(te_n))
print("дублей внутри train:", len(tr_n) - len(s_tr))
print("дублей внутри test:", len(te_n) - len(set(te_n)))

test, точно совпадающих с train: 3 из 12000
дублей внутри train: 1
дублей внутри test: 0


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(tr_n)
Xte = vec.transform(te_n)

best = np.zeros(len(te_n), dtype=np.float32)
nn = np.zeros(len(te_n), dtype=np.int64)
t0 = time.time()
for i in range(0, Xte.shape[0], 500):
    S = (Xte[i:i + 500] @ Xtr.T).toarray()
    nn[i:i + 500] = S.argmax(1)
    best[i:i + 500] = S.max(1)
print(f"готово за {time.time() - t0:.0f} с")

for th in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    m = best >= th
    same = (tr_y[nn[m]] == te_y[m]).mean() if m.any() else float("nan")
    print(f"косинус ≥ {th:.2f}: {m.mean():6.1%} test-заголовков | из них та же рубрика у соседа: {same:.1%}")

готово за 37 с
косинус ≥ 0.50:  25.9% test-заголовков | из них та же рубрика у соседа: 93.2%
косинус ≥ 0.60:  12.2% test-заголовков | из них та же рубрика у соседа: 95.8%
косинус ≥ 0.70:   5.4% test-заголовков | из них та же рубрика у соседа: 97.5%
косинус ≥ 0.80:   2.4% test-заголовков | из них та же рубрика у соседа: 98.3%
косинус ≥ 0.90:   0.6% test-заголовков | из них та же рубрика у соседа: 97.3%
косинус ≥ 0.95:   0.2% test-заголовков | из них та же рубрика у соседа: 96.0%


In [5]:
# Глазами посмотреть пары в разных диапазонах — по ним выбираем порог
for lo, hi in [(0.95, 1.01), (0.8, 0.95), (0.7, 0.8), (0.6, 0.7)]:
    idx = np.where((best >= lo) & (best < hi))[0]
    print(f"\n=== косинус в [{lo}, {hi}) — {len(idx)} шт. ===")
    for i in idx[:6]:
        print(f"  {best[i]:.2f}  TEST [{LABELS[te_y[i]]}] {te_text[i]}")
        print(f"        TRAIN [{LABELS[tr_y[nn[i]]]}] {tr_text[nn[i]]}")


=== косинус в [0.95, 1.01) — 25 шт. ===
  0.98  TEST [экономика] Рубль в начале дня вырос на 3 коп к доллару и упал на 3 коп к евро
        TRAIN [экономика] Рубль в начале дня вырос на 2 коп к доллару и упал на 2 коп к евро
  0.97  TEST [экономика] ММВБ СЭЛТ: Средневзвешенный курс доллара США по инструменту USDRUB_TOM на 11:30 мск - 55,8413 руб./долл
        TRAIN [экономика] ММВБ СЭЛТ: Средневзвешенный курс доллара США по инструменту USDRUB_TOM на 11:30 мск - 31,6414 руб./долл
  0.95  TEST [экономика] Средневзвешенный курс евро на ЕТС расчетами today повысился до 39,4762 руб/евро
        TRAIN [экономика] Средневзвешенный курс евро на ЕТС расчетами today повысился до 39,0003 руб/евро
  0.98  TEST [культура] В США объявлены претенденты на "Оскар"
        TRAIN [культура] Объявлены претенденты на "Оскар"
  1.00  TEST [культура] Букмекеры назвали фаворитов премии "Оскар"
        TRAIN [культура] Букмекеры назвали фаворитов премии Оскар
  0.96  TEST [экономика] Средневзвешенный курс дол

**Выбор порога.** Посмотри на пары выше: с какого косинуса это уже «тот же повод другими словами»? Порог ставится по примерам, а не наугад. Test-заголовки с соседом выше порога выбрасываем — остаётся «чистый» test, на нём и мерим.

Этот TF-IDF ловит почти-дословные совпадения. Парафразы совсем другими словами он пропустит — это ограничение проверки, его надо назвать в отчёте.

In [6]:
LEAK_TH = 0.8   # поменяй после просмотра примеров
clean_test = np.where(best < LEAK_TH)[0]
print(f"чистый test: {len(clean_test)} из {len(te_n)} ({len(clean_test) / len(te_n):.1%})")
print("рубрики в чистом test:", sorted(Counter(LABELS[c] for c in te_y[clean_test]).items()))

чистый test: 11710 из 12000 (97.6%)
рубрики в чистом test: [('культура', 1919), ('наука', 1942), ('политика', 1993), ('происшествия', 1972), ('спорт', 1989), ('экономика', 1895)]


## 3. Фиксируем выборки

Все варианты — TF-IDF, модель без примеров, с примерами, с LoRA — оцениваются **на одних и тех же 1200 заголовках** (по 200 на рубрику). Только так парное сравнение имеет смысл.

- **test:** 1200 из чистого test;
- **train для LoRA и для TF-IDF:** 3000 из train (по 500 на рубрику) — одинаковые данные для честного сравнения;
- **validation:** 300 — только чтобы следить за лоссом при обучении;
- **примеры для few-shot:** по одному на рубрику, из train вне обучающей выборки.

In [7]:
rng = random.Random(SEED)
va_y = np.array(ds["validation"]["label"])

def stratified(indices, y, per_class):
    out = []
    for c in range(6):
        pool = [int(i) for i in indices if y[i] == c]
        out += rng.sample(pool, per_class)
    rng.shuffle(out)
    return out

test_idx = stratified(clean_test, te_y, 200)
train_idx = stratified(range(len(tr_y)), tr_y, 500)
val_idx = stratified(range(len(va_y)), va_y, 50)
rest = sorted(set(range(len(tr_y))) - set(train_idx))
shot_idx = [rng.choice([i for i in rest if tr_y[i] == c]) for c in range(6)]

test_X = [te_text[i] for i in test_idx];  test_Y = [int(te_y[i]) for i in test_idx]
train_X = [tr_text[i] for i in train_idx]; train_Y = [int(tr_y[i]) for i in train_idx]
val_X = [ds["validation"][i]["text"] for i in val_idx]; val_Y = [int(va_y[i]) for i in val_idx]
SHOTS = [(tr_text[i], LABELS[tr_y[i]]) for i in shot_idx]

json.dump({"seed": SEED, "leak_threshold": LEAK_TH, "test_idx": test_idx, "train_idx": train_idx,
           "val_idx": val_idx, "shot_idx": shot_idx}, open("results/splits.json", "w"))
print(len(test_X), len(train_X), len(val_X), "| примеры для few-shot:")
for t, l in SHOTS:
    print(f"  [{l}] {t}")

1200 3000 300 | примеры для few-shot:
  [спорт] Рафаэль Надаль в 11-й раз в карьере вышел в финал "Ролан Гаррос"
  [происшествия] ЧМ-2009: США разгромили Францию
  [политика] 46 чеченцев добровольно сдались в плен под Горловкой, - боец батальона "Донбасс"
  [наука] СМИ сообщили о разработке "Фейсбуком" собственного модульного смартфона
  [культура] Никита Михалков готов покинуть Союз кинематографистов
  [экономика] "Почте России" разрешили заключать без торгов договоры аренды с "Лето Банком" - кабмин


In [8]:
cand_rng = random.Random(7)
for c in range(6):
    pool = [i for i in rest if tr_y[i] == c]
    print(f"\n=== {LABELS[c]} ===")
    for i in cand_rng.sample(pool, 6):
        print(f"  {i:6d}  {tr_text[i]}")


=== спорт ===
   17171  Футбол. Чемпионат Испании. Жозе Муринью: даже немного удивительно, что "Севилья" дважды поразили ворота Касильяса
    8015  Bare Knuckle FC. Артем Лобов - Пол Малиньяджи. Видео боя
   21068  Путин проведет тотальную проверку олимпийских объектов в Сочи
   34837  Российские легкоатлеты уличены в употреблении допинга
    2544  Сборная России упала на 19-е место в рейтинге ФИФА
    3830  Спринтерские гонки в Рупольдинге

=== происшествия ===
   28747  В Донецке восстановлено водоснабжение
    5263  Власти Египта обвиняют 250 членов Братьев-мусульман в убийствах
   19742  Осужденный за разбой инвалид Мамаев вернулся домой
   31045  СБ ООН: нападение на гей-клуб в Орландо - теракт
    3305  Число пострадавших в ДТП с участием автобуса в Кирове выросло до восьми человек
   27209  На Харьковщине 26 детей отравились в школьной столовой

=== политика ===
   11461  Обама и Кэмерон пригрозили России новыми санкциями
    1876  США заинтересованы в экономическом росте в Рос

In [9]:
shot_idx = [2544, 3305, 11461, 4871, 12015, 30946]
assert [tr_y[i] for i in shot_idx] == list(range(6))
SHOTS = [(tr_text[i], LABELS[tr_y[i]]) for i in shot_idx]

sp = json.load(open("results/splits.json"))
sp["shot_idx"] = shot_idx
sp["shot_selection"] = "ручной выбор однозначного примера из 6 случайных кандидатов на рубрику (seed 7)"
json.dump(sp, open("results/splits.json", "w"))
for t, l in SHOTS:
    print(f"  [{l}] {t}")

  [спорт] Сборная России упала на 19-е место в рейтинге ФИФА
  [происшествия] Число пострадавших в ДТП с участием автобуса в Кирове выросло до восьми человек
  [политика] Обама и Кэмерон пригрозили России новыми санкциями
  [наука] HTC Edge - первый 4-ядерный смартфон
  [культура] Началась работа над новым фильмом о The Beatles
  [экономика] ФАС озабочена ростом цен на дизтопливо


## 4. Метрики и бутстреп

- **accuracy — главная метрика.** Test сбалансирован (по 200 на рубрику), поэтому accuracy здесь равна сбалансированной полноте.
- **invalid** — доля ответов, которые не удалось сопоставить ни с одной рубрикой. Считаются ошибкой. Без примеров модель может часто отвечать не в формате, и тогда LoRA выиграет на формате, а не на понимании — поэтому и нужен few-shot как бейзлайн.
- **macro-F1 — только справочно.** При невалидных ответах он завышен: невалидный ответ снижает полноту своего класса, но не трогает точность ни одного класса, и модель, которая часто «не отвечает», получает F1 выше своей accuracy. На синтетике при 30% невалидных: accuracy 0.695, macro-F1 0.819.
- Доверительные интервалы и **парный бутстреп** по accuracy — все варианты отвечают на одни и те же заголовки.

In [10]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

def metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return {"acc": accuracy_score(y, p),
            "macro_f1": f1_score(y, p, labels=list(range(6)), average="macro", zero_division=0),
            "invalid": float(np.mean(p == -1))}

def bootstrap_ci(y, p, metric="acc", n=2000, seed=0):
    r = np.random.default_rng(seed); y, p = np.asarray(y), np.asarray(p)
    vals = [metrics(y[i], p[i])[metric] for i in (r.integers(0, len(y), len(y)) for _ in range(n))]
    return [float(v) for v in np.percentile(vals, [2.5, 97.5])]

def paired_bootstrap(y, pa, pb, metric="acc", n=2000, seed=0):
    """Разница a − b. p — доля перевыборок, где a не лучше b."""
    r = np.random.default_rng(seed); y, pa, pb = map(np.asarray, (y, pa, pb))
    d = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y))
        d.append(metrics(y[i], pa[i])[metric] - metrics(y[i], pb[i])[metric])
    d = np.array(d)
    return {"diff": metrics(y, pa)[metric] - metrics(y, pb)[metric],
            "ci": [float(v) for v in np.percentile(d, [2.5, 97.5])], "p": float(np.mean(d <= 0))}

RESULTS, PREDS = {}, {}
def report(name, preds):
    m = metrics(test_Y, preds)
    m["acc_ci"] = bootstrap_ci(test_Y, preds)
    RESULTS[name], PREDS[name] = m, [int(x) for x in preds]
    print(f"{name:28s} acc {m['acc']:.3f} [{m['acc_ci'][0]:.3f}; {m['acc_ci'][1]:.3f}] "
          f"| invalid {m['invalid']:5.1%} | macro-F1 {m['macro_f1']:.3f} (справочно)")

## 5. Бейзлайн без нейросетей: TF-IDF + логистическая регрессия

Слова (1–2-граммы) плюс символьные 3–5-граммы. Обучаем **на тех же 3000**, что и LoRA. Отдельно — на всём train как ориентир, сколько дают все данные.

Гиперпараметры не подбираем на test.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union

def tfidf_lr():
    return make_pipeline(
        make_union(TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True),
                   TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
        LogisticRegression(max_iter=3000))

clf = tfidf_lr().fit(train_X, train_Y)
report("TF-IDF+LR (3000)", clf.predict(test_X))

clf_full = tfidf_lr().fit(tr_text, tr_y)
report("TF-IDF+LR (весь train)", clf_full.predict(test_X))

TF-IDF+LR (3000)             acc 0.810 [0.787; 0.833] | invalid  0.0% | macro-F1 0.810 (справочно)
TF-IDF+LR (весь train)       acc 0.877 [0.858; 0.896] | invalid  0.0% | macro-F1 0.877 (справочно)


## 6. Модель без дообучения

Ответ генерируется жадно, до 6 токенов, приводится к нижнему регистру и сопоставляется с рубрикой. Не сопоставилось — `invalid`.

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")
model.eval()

SYSTEM = ("Ты классифицируешь заголовки русских новостей по рубрикам. "
          "Ответь одним словом из списка:\n"
          "спорт — соревнования, команды, спортсмены;\n"
          "происшествия — ДТП, катастрофы, преступления, теракты, стихийные бедствия;\n"
          "политика — власть, выборы, международные отношения, санкции;\n"
          "наука — наука, технологии, космос, гаджеты;\n"
          "культура — кино, музыка, премии, знаменитости;\n"
          "экономика — рынки, курсы валют, банки, цены, компании.")
LABEL2ID = {l: i for i, l in enumerate(LABELS)}

def build_messages(text, shots=()):
    msgs = [{"role": "system", "content": SYSTEM}]
    for t, l in shots:
        msgs += [{"role": "user", "content": f"Заголовок: {t}"}, {"role": "assistant", "content": l}]
    msgs.append({"role": "user", "content": f"Заголовок: {text}"})
    return msgs

def to_prompt(msgs):
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse(s):
    w = re.sub(r"[^\w\s]", " ", s.lower().replace("ё", "е")).split()
    if not w:
        return -1
    if w[0] in LABEL2ID:
        return LABEL2ID[w[0]]
    for l, i in LABEL2ID.items():          # «происшествие», «политики» и т.п.
        if len(w[0]) >= 4 and l.startswith(w[0][:4]):
            return i
    return -1

@torch.no_grad()
def predict(m, texts, shots=(), bs=32, max_new_tokens=6):
    preds, raws = [], []
    for i in range(0, len(texts), bs):
        prompts = [to_prompt(build_messages(t, shots)) for t in texts[i:i + bs]]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
        out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        gen = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        raws += gen
        preds += [parse(g) for g in gen]
    return preds, raws

p, raw = predict(model, test_X[:8])     # проверка на 8 примерах
for t, y, r in zip(test_X[:8], test_Y[:8], raw):
    print(f"[{LABELS[y]:12s}] → {r!r:20s} | {t}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[наука       ] → 'политика'           | НАСА обещает помочь SpaceX с запуском туристов в полет вокруг Луны
[культура    ] → 'политика'           | Стругацкий обвинил режиссера "Аватара" в плагиате
[наука       ] → 'спорте'             | Ребенок попытался купить на онлайн-аукционе реактивный истребитель
[политика    ] → 'политика'           | Путин рассказал о поддержке боевиков Северного Кавказа спецслужбами США
[культура    ] → 'политика'           | Чеченский писатель требует миллиард от Джеймса Кэмерона, обвиняя того в плагиате
[происшествия] → 'спорте'             | Пограничники применили оружие при задержании браконьеров на Каспии
[происшествия] → 'происшествия'       | В Киевской области ограбили ювелирный магазин
[спорт       ] → 'спорта'             | Тренер сборной Финляндии по хоккею Юкка Ялонен будет тренировать СКА


In [13]:
t0 = time.time()
p_zero, raw_zero = predict(model, test_X)
print(f"{time.time() - t0:.0f} с")
report("Qwen zero-shot", p_zero)
print("примеры невалидных ответов:", [r for r, x in zip(raw_zero, p_zero) if x == -1][:10])

58 с
Qwen zero-shot               acc 0.408 [0.381; 0.435] | invalid 24.2% | macro-F1 0.416 (справочно)
примеры невалидных ответов: ['soccer', 'эко', 'судно', 'politica', ' culture', 'sport', 'соревнования,', 'соревнования,', 'politica', 'soccer']


In [14]:
t0 = time.time()
p_few, raw_few = predict(model, test_X, shots=SHOTS)
print(f"{time.time() - t0:.0f} с")
report("Qwen few-shot (6 примеров)", p_few)
print("примеры невалидных ответов:", [r for r, x in zip(raw_few, p_few) if x == -1][:10])

118 с
Qwen few-shot (6 примеров)   acc 0.689 [0.662; 0.716] | invalid  8.6% | macro-F1 0.697 (справочно)
примеры невалидных ответов: ['космос', 'технологии', 'технологии', 'culture', ' medicine', 'culture', 'culture', 'коммуникации', 'технологии', 'технологии']


In [15]:
# Апостериорный мягкий разбор: только диагностика «формат или понимание».
# Главная метрика — строгий разбор, зафиксированный до запуска.
SYNONYMS = {
    "спорт": ["sport", "sports", "soccer", "football", "hockey", "соревнования", "футбол", "хоккей", "теннис"],
    "происшествия": ["incident", "incidents", "accident", "crime", "происшествие", "криминал", "катастрофа", "дтп"],
    "политика": ["politics", "politica", "political", "policy"],
    "наука": ["science", "technology", "technologies", "tech", "технологии", "технология", "космос",
              "space", "medicine", "медицина", "гаджеты"],
    "культура": ["culture", "cultura", "кино", "музыка", "art", "entertainment"],
    "экономика": ["economy", "economics", "economic", "business", "finance", "эко", "финансы", "бизнес"],
}
SYN2ID = {w: LABEL2ID[l] for l, ws in SYNONYMS.items() for w in ws}

def parse_lenient(s):
    p = parse(s)
    if p != -1:
        return p
    w = re.sub(r"[^\w\s]", " ", s.lower().replace("ё", "е")).split()
    return SYN2ID.get(w[0], -1) if w else -1

for name, raws in [("Qwen zero-shot", raw_zero), ("Qwen few-shot (6 примеров)", raw_few)]:
    report(name + " [мягкий разбор]", [parse_lenient(r) for r in raws])

json.dump({"zero": raw_zero, "few": raw_few}, open("results/raw_outputs.json", "w"), ensure_ascii=False)
print("сырые ответы сохранены в results/raw_outputs.json")

Qwen zero-shot [мягкий разбор] acc 0.503 [0.475; 0.532] | invalid 10.7% | macro-F1 0.493 (справочно)
Qwen few-shot (6 примеров) [мягкий разбор] acc 0.762 [0.738; 0.787] | invalid  0.9% | macro-F1 0.763 (справочно)
сырые ответы сохранены в results/raw_outputs.json


In [16]:
json.dump({"results": RESULTS, "preds": PREDS, "test_Y": test_Y}, open("results/baselines.json", "w"),
          ensure_ascii=False, indent=1)
print("сохранено results/baselines.json")

сохранено results/baselines.json


---
## ⏸ Точка остановки

**Пришли мне вывод разделов 1–6** — особенно проверку утечки и таблицу бейзлайнов. Обсудим, прежде чем обучать: если few-shot уже близок к потолку TF-IDF на всём train, это меняет постановку.

Файлы `results/` скачай себе — Colab стирает их при отключении.
---

## 7. LoRA: подготовка данных

Пример для обучения — тот же диалог, что в zero-shot, плюс ответ модели: рубрика и маркер конца реплики `<|im_end|>`, чтобы модель училась останавливаться.

**Лосс считается только по ответу.** На токенах промпта метка `-100` — они не участвуют в обучении. Иначе модель тратила бы ёмкость на запоминание системной инструкции и заголовков.

In [17]:
END = "<|im_end|>"

def make_example(text, label_id):
    p_ids = tok(to_prompt(build_messages(text)), add_special_tokens=False)["input_ids"]
    a_ids = tok(LABELS[label_id] + END, add_special_tokens=False)["input_ids"]
    return {"input_ids": p_ids + a_ids, "labels": [-100] * len(p_ids) + a_ids}

train_ds = [make_example(t, y) for t, y in zip(train_X, train_Y)]
val_ds = [make_example(t, y) for t, y in zip(val_X, val_Y)]
lens = [len(e["input_ids"]) for e in train_ds]
print("длина примера в токенах: медиана", int(np.median(lens)), "| максимум", max(lens))
print("токены ответа в первом примере:", tok.convert_ids_to_tokens([x for x in train_ds[0]["labels"] if x != -100]))

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), n), tok.pad_token_id)
    lab = torch.full((len(batch), n), -100)
    att = torch.zeros((len(batch), n), dtype=torch.long)
    for j, b in enumerate(batch):
        k = len(b["input_ids"])
        ids[j, :k] = torch.tensor(b["input_ids"])
        lab[j, :k] = torch.tensor(b["labels"])
        att[j, :k] = 1
    return {"input_ids": ids, "labels": lab, "attention_mask": att}

длина примера в токенах: медиана 204 | максимум 261
токены ответа в первом примере: ['Ð½Ð°', 'Ñĥ', 'ÐºÐ°', '<|im_end|>']


## 8. LoRA: конфигурация

- **`r=16`** — ранг добавки. Вес меняется не целиком, а на произведение двух матриц размером `d×16` и `16×d`. Больше ранг — больше ёмкость и параметров.
- **`lora_alpha=32`** — масштаб добавки, фактический множитель `alpha / r = 2`.
- **`lora_dropout=0.05`** — регуляризация на входе адаптера.
- **`target_modules`** — проекции внимания `q, k, v, o`. Можно добавить полносвязные `gate, up, down` — больше параметров, иногда лучше.
- Базовые веса **заморожены** и остаются в fp16. Обучаемые параметры адаптера переводим в fp32 — иначе смешанная точность на T4 падает при масштабировании градиентов.

In [18]:
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM")
model.config.use_cache = False
pm = get_peft_model(model, lora_cfg)
for prm in pm.parameters():
    if prm.requires_grad:
        prm.data = prm.data.float()
pm.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


## 9. LoRA: обучение

3000 примеров, 2 эпохи, эффективный батч 16 (микробатч 4 × накопление 4) → 376 шагов. `lr=2e-4` с косинусным затуханием и разогревом — типичная отправная точка для LoRA. Следим за лоссом на validation по эпохам: если он растёт, а train падает — переобучение.

In [19]:
import gc
from transformers import Trainer, TrainingArguments

# освобождаем память от упавшей попытки
if "trainer" in globals():
    del trainer
pm.zero_grad(set_to_none=True)
gc.collect()
torch.cuda.empty_cache()
print(f"занято на GPU перед обучением: {torch.cuda.memory_allocated() / 1e9:.1f} ГБ")

args = TrainingArguments(
    output_dir="lora_out",
    # логиты 16 × ~260 × 152k словаря в fp32 ≈ 2.5 ГБ не влезают в T4 → батч 4 × накопление 4 = эффективный 16
    per_device_train_batch_size=4, gradient_accumulation_steps=4, per_device_eval_batch_size=4,
    num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="cosine",
    warmup_steps=11,  # 3% от 376 шагов; warmup_ratio убран в transformers 5
    logging_steps=25, eval_strategy="epoch", save_strategy="no",
    fp16=True, report_to="none", seed=SEED, remove_unused_columns=False)

trainer = Trainer(model=pm, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate)
t0 = time.time()
trainer.train()
print(f"обучение: {(time.time() - t0) / 60:.1f} мин")
pm.save_pretrained("results/lora_adapter")

for rec in trainer.state.log_history:
    if "loss" in rec or "eval_loss" in rec:
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in rec.items()
               if k in ("step", "epoch", "loss", "eval_loss")})

занято на GPU перед обучением: 3.1 ГБ


Epoch,Training Loss,Validation Loss
1,0.111267,0.126035
2,0.065031,0.109659


обучение: 12.3 мин
{'loss': 0.3006, 'epoch': 0.1333, 'step': 25}
{'loss': 0.1192, 'epoch': 0.2667, 'step': 50}
{'loss': 0.1559, 'epoch': 0.4, 'step': 75}
{'loss': 0.1273, 'epoch': 0.5333, 'step': 100}
{'loss': 0.1436, 'epoch': 0.6667, 'step': 125}
{'loss': 0.1507, 'epoch': 0.8, 'step': 150}
{'loss': 0.1113, 'epoch': 0.9333, 'step': 175}
{'eval_loss': 0.126, 'epoch': 1.0, 'step': 188}
{'loss': 0.1051, 'epoch': 1.064, 'step': 200}
{'loss': 0.0726, 'epoch': 1.1973, 'step': 225}
{'loss': 0.0873, 'epoch': 1.3307, 'step': 250}
{'loss': 0.0909, 'epoch': 1.464, 'step': 275}
{'loss': 0.0856, 'epoch': 1.5973, 'step': 300}
{'loss': 0.0886, 'epoch': 1.7307, 'step': 325}
{'loss': 0.0879, 'epoch': 1.864, 'step': 350}
{'loss': 0.065, 'epoch': 1.9973, 'step': 375}
{'eval_loss': 0.1097, 'epoch': 2.0, 'step': 376}


## 10. Оценка LoRA — на тех же 1200 заголовках

In [20]:
pm.eval(); pm.config.use_cache = True
t0 = time.time()
p_lora, raw_lora = predict(pm, test_X)
print(f"{time.time() - t0:.0f} с")
report("Qwen + LoRA", p_lora)
print("примеры невалидных ответов:", [r for r, x in zip(raw_lora, p_lora) if x == -1][:10])

74 с
Qwen + LoRA                  acc 0.885 [0.867; 0.902] | invalid  0.0% | macro-F1 0.885 (справочно)
примеры невалидных ответов: []


In [21]:
print("Парный бутстреп по accuracy (a − b):")
for a, b in [("Qwen + LoRA", "Qwen few-shot (6 примеров)"),
             ("Qwen + LoRA", "TF-IDF+LR (3000)"),
             ("Qwen + LoRA", "TF-IDF+LR (весь train)"),
             ("Qwen few-shot (6 примеров)", "Qwen zero-shot")]:
    r = paired_bootstrap(test_Y, PREDS[a], PREDS[b])
    print(f"  {a} − {b}: {r['diff']:+.3f} [{r['ci'][0]:+.3f}; {r['ci'][1]:+.3f}], p = {r['p']:.3f}")

print("\nМатрица ошибок LoRA (строки — истина, столбцы — ответ; последний столбец — invalid):")
cm = confusion_matrix(test_Y, p_lora, labels=list(range(6)) + [-1])[:6]
print(" " * 14 + " ".join(f"{l[:5]:>6s}" for l in LABELS) + "  inval")
for l, row in zip(LABELS, cm):
    print(f"{l:14s}" + " ".join(f"{v:6d}" for v in row))

Парный бутстреп по accuracy (a − b):
  Qwen + LoRA − Qwen few-shot (6 примеров): +0.196 [+0.170; +0.222], p = 0.000
  Qwen + LoRA − TF-IDF+LR (3000): +0.075 [+0.054; +0.096], p = 0.000
  Qwen + LoRA − TF-IDF+LR (весь train): +0.008 [-0.010; +0.027], p = 0.188
  Qwen few-shot (6 примеров) − Qwen zero-shot: +0.281 [+0.251; +0.309], p = 0.000

Матрица ошибок LoRA (строки — истина, столбцы — ответ; последний столбец — invalid):
               спорт  проис  полит  наука  культ  эконо  inval
спорт            195      1      1      0      3      0      0
происшествия       1    177     12      3      2      5      0
политика           1      5    176      2      8      8      0
наука              4      4      2    177      6      7      0
культура          17      2      6      6    166      3      0
экономика          1      3     12     10      3    171      0


In [22]:
json.dump({"results": RESULTS, "preds": PREDS, "test_Y": test_Y,
           "lora": {"r": 16, "alpha": 32, "dropout": 0.05, "targets": ["q_proj", "k_proj", "v_proj", "o_proj"],
                    "epochs": 2, "lr": 2e-4, "batch": 16, "train_size": len(train_X)},
           "log_history": trainer.state.log_history},
          open("results/final.json", "w"), ensure_ascii=False, indent=1)
print("сохранено results/final.json")

сохранено results/final.json


## 11. Не разучилась ли модель общим вещам (быстрая проверка)

Те же вопросы — с выключенным адаптером и с включённым. Ничего не меряем, просто смотрим: если на общие вопросы модель с адаптером отвечает одним словом-рубрикой, она переобучилась на формат.

In [23]:
QUESTIONS = ["Объясни в двух предложениях, что такое инфляция.",
             "Как перевести на английский фразу «я опоздал на поезд»?",
             "Назови три столицы европейских стран."]

@torch.no_grad()
def chat(m, q):
    p = tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
    enc = tok(p, return_tensors="pt", add_special_tokens=False).to(m.device)
    out = m.generate(**enc, max_new_tokens=80, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

for q in QUESTIONS:
    with pm.disable_adapter():
        base = chat(pm, q)
    print(f"Q: {q}\n  без адаптера: {base}\n  с адаптером:  {chat(pm, q)}\n")

Q: Объясни в двух предложениях, что такое инфляция.
  без адаптера: Инфляция - это рост цен на товары и услуги в экономике за определенный период времени. Это происходит из-за увеличения объема выпускаемых товаров и услуг или из-за уменьшения производительности труда.

Второе предложение: Инфляция может быть классифицирована как "инфля
  с адаптером:  Инфляция - это рост цен на товары и услуги в экономике за определенный период времени. Это происходит из-за увеличения объема денег или избыточного спроса на продукты. В результате цены на товары и услуги повышаются, что может привести к снижению покупательской способности населения.

Втор

Q: Как перевести на английский фразу «я опоздал на поезд»?
  без адаптера: Фраза "Я опоздал на поезд" переводится на английский язык как "I was late for the train".
  с адаптером:  The English translation of "я опоздал на поезд" is "I missed the train."

Q: Назови три столицы европейских стран.
  без адаптера: Три столицы европейских стран:

1. Париж (

In [24]:
r = paired_bootstrap(test_Y, PREDS["Qwen + LoRA"], PREDS["Qwen few-shot (6 примеров) [мягкий разбор]"])
print(f"LoRA − few-shot [мягкий]: {r['diff']:+.3f} [{r['ci'][0]:+.3f}; {r['ci'][1]:+.3f}], p = {r['p']:.3f}")

LoRA − few-shot [мягкий]: +0.123 [+0.100; +0.145], p = 0.000


In [25]:
errs = [i for i, (y, p) in enumerate(zip(test_Y, p_lora)) if y != p]
print(f"ошибок: {len(errs)}\n")

print("=== культура → спорт ===")
for i in errs:
    if test_Y[i] == 4 and p_lora[i] == 0:
        print("  ", test_X[i])

print("\n=== случайные 40 ошибок ===")
for k, i in enumerate(random.Random(0).sample(errs, 40)):
    print(f"{k:2d} [{LABELS[test_Y[i]]} → {LABELS[p_lora[i]]}] {test_X[i]}")

ошибок: 138

=== культура → спорт ===
   Харден продлит контракт с "Хьюстоном", его зарплата составит 118 миллионов долларов
   Стивен Джеррард не сыграет с Нидерландами
   Выступить в Туле Расторгуеву помешала возрастная аритмия
   Автор легендарной фразы о "всегда побеждающих в футбол немцах" снова переделал ее
   Ребров: "Приехали, чтобы постараться выиграть"
   В последней гонке кубкового сезона Дарья Домрачева стала четвертой и впервые в карьере завоевала Большой хрустальный глобус
   "Труа" сыграл вничью с "Лорьяном" и вышел в лигу 1
   У звезды датского футбола Брайана Лаудрупа обнаружен рак
   Английская Премьер-лига отказала в присоединении к ней шотландских клубов
   Хабиб назвал Конора ревнивой женой
   Месси и Роналду такое даже не снилось: форвард "Ромы" установил уникальное достижение в топ-лигах
   "Таганка" осталась без Любимова
   Южноафриканский пловец выиграл золото с мировым рекордом
   Александр Осипов: "Все впереди: и пот, и кровь, и усталость"
   На открытии Олим